# Visualizacao dos Resultados de Classificacao

Este notebook carrega os CSVs ja gerados pela pasta `classificacao/outputs` e organiza tabelas e graficos para comparar os modelos. Ele nao reroda os treinos nem os testes; usa apenas os arquivos salvos.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib nao esta instalado; as tabelas serao exibidas e os graficos serao pulados.")

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)

if plt is not None:
    plt.style.use("seaborn-v0_8-whitegrid")

## 1. Carregamento dos CSVs

In [ ]:
candidate_dirs = [
    Path("outputs"),
    Path("classificacao/outputs"),
    Path("EstresseHidricoFinal/classificacao/outputs"),
]

BASE_DIR = next((path for path in candidate_dirs if path.exists()), None)
if BASE_DIR is None:
    raise FileNotFoundError("Nao encontrei a pasta outputs da classificacao.")

print(f"Pasta de resultados: {BASE_DIR.resolve()}")

def ler_csvs(nome_arquivo):
    arquivos = sorted(BASE_DIR.rglob(nome_arquivo))
    tabelas = []
    for arquivo in arquivos:
        df = pd.read_csv(arquivo, sep=";")
        df["arquivo_origem"] = str(arquivo.relative_to(BASE_DIR))
        tabelas.append(df)
    if not tabelas:
        return pd.DataFrame()
    return pd.concat(tabelas, ignore_index=True)

metricas_resumo = ler_csvs("metricas_resumo.csv")
metricas_folds = ler_csvs("metricas_folds.csv")
predicoes = ler_csvs("predicoes.csv")
matrizes = ler_csvs("matriz_confusao_folds.csv")
curvas_roc = ler_csvs("curva_roc_folds.csv")

importancia_rf = ler_csvs("importancia_bandas.csv")
importancia_svm = ler_csvs("importancia_permutacao.csv")
coeficientes_pls = ler_csvs("coeficientes_pls.csv")
pesos_pls = ler_csvs("pesos_pls.csv")

print(f"metricas_resumo: {metricas_resumo.shape}")
print(f"metricas_folds: {metricas_folds.shape}")
print(f"predicoes: {predicoes.shape}")
print(f"matrizes: {matrizes.shape}")
print(f"curvas_roc: {curvas_roc.shape}")

## 2. Tabelas de metricas

In [ ]:
metric_cols_media = [
    "accuracy_media", "precision_media", "recall_media",
    "f1_score_media", "kappa_media", "auc_roc_media",
]
metric_cols_desvio = [col.replace("_media", "_desvio") for col in metric_cols_media]

metricas_resumo_ordenado = metricas_resumo.sort_values(
    ["modo", "genotipo", "accuracy_media"], ascending=[True, True, False]
).reset_index(drop=True)

metricas_resumo_ordenado[
    ["modelo", "modo", "genotipo", *metric_cols_media, *metric_cols_desvio]
]

In [ ]:
ranking_accuracy = metricas_resumo.sort_values("accuracy_media", ascending=False)[
    ["modelo", "modo", "genotipo", "accuracy_media", "accuracy_desvio", "auc_roc_media", "f1_score_media"]
].reset_index(drop=True)

ranking_auc = metricas_resumo.sort_values("auc_roc_media", ascending=False)[
    ["modelo", "modo", "genotipo", "auc_roc_media", "auc_roc_desvio", "accuracy_media", "f1_score_media"]
].reset_index(drop=True)

display(ranking_accuracy)
display(ranking_auc)

## 3. Graficos comparativos das metricas medias

In [ ]:
def plot_metricas_por_cenario(df, metrica="accuracy_media"):
    if plt is None:
        print("Grafico pulado: instale matplotlib para visualizar.")
        return
    if df.empty:
        print("Sem dados para plotar.")
        return

    dados = df.copy()
    dados["cenario"] = dados["modo"] + " | " + dados["genotipo"].astype(str)
    cenarios = dados["cenario"].drop_duplicates().tolist()
    modelos = dados["modelo"].drop_duplicates().tolist()

    x = np.arange(len(cenarios))
    width = 0.8 / max(len(modelos), 1)

    fig, ax = plt.subplots(figsize=(max(10, len(cenarios) * 1.6), 5))
    for i, modelo in enumerate(modelos):
        sub = dados[dados["modelo"] == modelo].set_index("cenario").reindex(cenarios)
        ax.bar(x + (i - (len(modelos) - 1) / 2) * width, sub[metrica], width, label=modelo)

    ax.set_title(metrica.replace("_", " ").title())
    ax.set_ylabel(metrica)
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x)
    ax.set_xticklabels(cenarios, rotation=35, ha="right")
    fig.legend(loc="center left", bbox_to_anchor=(0.81, 0.5), frameon=False, title="Modelo")
    plt.tight_layout(rect=(0, 0, 0.80, 1))
    plt.show()

for metrica in ["accuracy_media", "precision_media", "recall_media", "f1_score_media", "kappa_media", "auc_roc_media"]:
    plot_metricas_por_cenario(metricas_resumo, metrica)

## 4. Variabilidade entre folds

In [ ]:
metricas_folds.sort_values(["modo", "genotipo", "modelo", "fold"]).reset_index(drop=True)

In [ ]:
def plot_variabilidade_folds(df, metrica="accuracy"):
    if plt is None:
        print("Grafico pulado: instale matplotlib para visualizar.")
        return
    if df.empty:
        print("Sem dados para plotar.")
        return

    dados = df.copy()
    cores = {
        "gradient_boosting": "#E69F00",
        "pls_da": "#009E73",
        "random_forest": "#0072B2",
        "svm": "#CC79A7",
    }

    # Cenarios sem recorte diario: uma linha por modo, genotipo e modelo.
    gerais = dados.loc[dados["modo"] != "por_dia"].copy()
    if not gerais.empty:
        resumo = (
            gerais.groupby(["modo", "genotipo", "modelo"], sort=False)[metrica]
            .agg(media="mean", minimo="min", maximo="max")
            .reset_index()
        )
        resumo["cenario"] = (
            resumo["modo"] + " | " + resumo["genotipo"].astype(str)
            + " | " + resumo["modelo"]
        )
        y = np.arange(len(resumo))
        fig, ax = plt.subplots(figsize=(12, max(5, 0.45 * len(resumo))))
        for modelo, grupo in resumo.groupby("modelo", sort=False):
            pos = grupo.index.to_numpy()
            ax.errorbar(
                grupo["media"], pos,
                xerr=[grupo["media"] - grupo["minimo"],
                      grupo["maximo"] - grupo["media"]],
                fmt="o", capsize=4, markersize=6, linewidth=1.5,
                color=cores.get(modelo, "#555555"), label=modelo,
            )
        ax.set_yticks(y, resumo["cenario"])
        ax.set_xlim(0, 1.05)
        ax.set_xlabel(metrica)
        ax.set_title(f"Variabilidade de {metrica}: media e intervalo dos folds")
        ax.invert_yaxis()
        ax.legend(title="Modelo", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.show()

    # Cenarios diarios: nao mistura a variacao entre dias com a dos folds.
    diarios = dados.loc[dados["modo"] == "por_dia"].copy()
    if not diarios.empty and "dia" in diarios.columns:
        diarios = diarios.dropna(subset=["dia"])
        resumo_dia = (
            diarios.groupby(["genotipo", "dia", "modelo"], sort=True)[metrica]
            .agg(media="mean", minimo="min", maximo="max")
            .reset_index()
        )
        genotipos = resumo_dia["genotipo"].drop_duplicates().tolist()
        fig, axes = plt.subplots(
            len(genotipos), 1, figsize=(12, 3.8 * len(genotipos)),
            sharex=True, sharey=True, squeeze=False,
        )
        modelos = resumo_dia["modelo"].drop_duplicates().tolist()
        deslocamentos = np.linspace(-0.12, 0.12, len(modelos))
        dias = sorted(resumo_dia["dia"].drop_duplicates())
        mapa_dias = {dia: i for i, dia in enumerate(dias)}
        for ax, genotipo in zip(axes.flat, genotipos):
            sub_genotipo = resumo_dia.loc[resumo_dia["genotipo"] == genotipo]
            for deslocamento, modelo in zip(deslocamentos, modelos):
                grupo = sub_genotipo.loc[sub_genotipo["modelo"] == modelo]
                if grupo.empty:
                    continue
                x = grupo["dia"].map(mapa_dias).to_numpy() + deslocamento
                ax.errorbar(
                    x, grupo["media"],
                    yerr=[grupo["media"] - grupo["minimo"],
                          grupo["maximo"] - grupo["media"]],
                    fmt="o-", capsize=3, markersize=5, linewidth=1.4,
                    color=cores.get(modelo, "#555555"), label=modelo,
                )
            ax.set_title(str(genotipo), loc="left", weight="bold")
            ax.set_ylabel(metrica)
            ax.set_ylim(0, 1.05)
        axes[-1, 0].set_xticks(range(len(dias)), dias)
        axes[-1, 0].set_xlabel("Dia")
        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.suptitle(
            f"Variabilidade de {metrica} por dia: media e intervalo dos folds",
            y=0.995,
        )
        fig.legend(
            handles, labels, title="Modelo", loc="upper center",
            bbox_to_anchor=(0.5, 0.965), ncol=len(modelos),
        )
        plt.tight_layout(rect=(0, 0, 1, 0.91))
        plt.show()

plot_variabilidade_folds(metricas_folds, "f1_score")
#plot_variabilidade_folds(metricas_folds, "auc_roc")

## 5. Matrizes de confusao

In [ ]:
matriz_total = (
    matrizes.groupby(["modelo", "modo", "genotipo", "real", "predito"], as_index=False)["n"].sum()
    .sort_values(["modo", "genotipo", "modelo", "real", "predito"])
)
matriz_total

In [ ]:
def plot_matrizes_confusao(df):
    if plt is None:
        print("Grafico pulado: instale matplotlib para visualizar.")
        return
    if df.empty:
        print("Sem dados para plotar.")
        return

    cenarios = df[["modelo", "modo", "genotipo"]].drop_duplicates().to_dict("records")
    n = len(cenarios)
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.2, rows * 3.8))
    axes = np.array(axes).reshape(-1)

    for ax, cenario in zip(axes, cenarios):
        sub = df[
            (df["modelo"] == cenario["modelo"])
            & (df["modo"] == cenario["modo"])
            & (df["genotipo"] == cenario["genotipo"])
        ]
        tabela = sub.pivot(index="real", columns="predito", values="n").fillna(0)
        labels = sorted(set(tabela.index).union(tabela.columns))
        tabela = tabela.reindex(index=labels, columns=labels, fill_value=0)

        im = ax.imshow(tabela.values, cmap="Blues")
        ax.set_title(f"{cenario['modo']} | {cenario['genotipo']} | {cenario['modelo']}")
        ax.set_xlabel("Predito")
        ax.set_ylabel("Real")
        ax.set_xticks(np.arange(len(labels)))
        ax.set_yticks(np.arange(len(labels)))
        ax.set_xticklabels(labels)
        ax.set_yticklabels(labels)
        for i in range(len(labels)):
            for j in range(len(labels)):
                ax.text(j, i, int(tabela.values[i, j]), ha="center", va="center", color="black")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    for ax in axes[n:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

plot_matrizes_confusao(matriz_total)

## 6. Curvas ROC por fold

In [ ]:
curvas_roc.head()

In [ ]:
def plot_curvas_roc(df, modo=None, genotipo=None):
    if plt is None:
        print("Grafico pulado: instale matplotlib para visualizar.")
        return
    if df.empty:
        print("Sem dados para plotar.")
        return

    dados = df.copy()
    if modo is not None:
        dados = dados[dados["modo"] == modo]
    if genotipo is not None:
        dados = dados[dados["genotipo"] == genotipo]

    if dados.empty:
        print("Sem curvas para o filtro escolhido.")
        return

    cenarios = dados[["modelo", "modo", "genotipo"]].drop_duplicates().to_dict("records")
    n = len(cenarios)
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.6, rows * 4.0))
    axes = np.array(axes).reshape(-1)

    for ax, cenario in zip(axes, cenarios):
        sub = dados[
            (dados["modelo"] == cenario["modelo"])
            & (dados["modo"] == cenario["modo"])
            & (dados["genotipo"] == cenario["genotipo"])
        ]
        for fold, fold_df in sub.groupby("fold"):
            ax.plot(fold_df["fpr"], fold_df["tpr"], alpha=0.45, label=f"fold {fold}")
        ax.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1)
        ax.set_title(f"{cenario['modo']} | {cenario['genotipo']} | {cenario['modelo']}")
        ax.set_xlabel("FPR")
        ax.set_ylabel("TPR")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.02)

    for ax in axes[n:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

plot_curvas_roc(curvas_roc, modo="agrupado")
plot_curvas_roc(curvas_roc, modo="por_genotipo")

## 7. Importancia das bandas e coeficientes

In [ ]:
display(importancia_rf.head(20))
display(importancia_svm.head(20))
display(coeficientes_pls.head(20))

In [ ]:
def plot_importancias(df, valor_col, titulo):
    if plt is None:
        print("Grafico pulado: instale matplotlib para visualizar.")
        return
    if df.empty:
        print(f"Sem dados para {titulo}.")
        return

    dados = df.copy()
    dados["cenario"] = dados["modo"] + " | " + dados["genotipo"].astype(str) + " | fold " + dados["fold"].astype(str)
    resumo = (
        dados.groupby(["modelo", "modo", "genotipo", "banda_nm"], as_index=False)[valor_col]
        .mean()
        .sort_values(["modo", "genotipo", valor_col], ascending=[True, True, False])
    )

    cenarios = resumo[["modelo", "modo", "genotipo"]].drop_duplicates().to_dict("records")
    n = len(cenarios)
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.8, rows * 3.8))
    axes = np.array(axes).reshape(-1)

    for ax, cenario in zip(axes, cenarios):
        sub = resumo[
            (resumo["modelo"] == cenario["modelo"])
            & (resumo["modo"] == cenario["modo"])
            & (resumo["genotipo"] == cenario["genotipo"])
        ]
        ax.bar(sub["banda_nm"].astype(str), sub[valor_col])
        ax.set_title(f"{cenario['modo']} | {cenario['genotipo']}")
        ax.set_xlabel("Banda (nm)")
        ax.set_ylabel(valor_col)
        ax.tick_params(axis="x", rotation=45)

    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(titulo, y=1.02)
    plt.tight_layout()
    plt.show()

plot_importancias(importancia_rf, "importancia", "Importancia das bandas - Random Forest")
plot_importancias(importancia_svm, "importancia_media", "Importancia por permutacao - SVM")
plot_importancias(coeficientes_pls.assign(coef_abs=coeficientes_pls.get("coeficiente", pd.Series(dtype=float)).abs()), "coef_abs", "Coeficientes absolutos - PLS-DA")

## 8. Leituras rapidas

In [ ]:
if metricas_resumo.empty:
    print("Nao ha metricas carregadas.")
else:
    melhor_acc = metricas_resumo.loc[metricas_resumo["accuracy_media"].idxmax()]
    melhor_auc = metricas_resumo.loc[metricas_resumo["auc_roc_media"].idxmax()]

    print("Melhor accuracy media:")
    print(f"  {melhor_acc['modelo']} | {melhor_acc['modo']} | {melhor_acc['genotipo']} = {melhor_acc['accuracy_media']:.4f}")
    print("Melhor AUC-ROC media:")
    print(f"  {melhor_auc['modelo']} | {melhor_auc['modo']} | {melhor_auc['genotipo']} = {melhor_auc['auc_roc_media']:.4f}")

    print("\nModelos/cenarios avaliados:")
    display(metricas_resumo[["modelo", "modo", "genotipo"]].drop_duplicates().reset_index(drop=True))